# 3D-ReVert — Adapted for Lumbar Inter-Comparison with PPC v12.3

## What changed vs original 3D-ReVert
| Component | Change |
|---|---|
| Dataset loader | New `LumbarDataset` — reads AP + LP DRRs from Lumbar_Filtered_1037 |
| Encoder input | `in_channels` 1 → **2** (AP + LP concatenated as 2-channel image) |
| `num_points` | Set to **8192** to match PPC v12.3 |
| Loss | **Unchanged** — 3D-ReVert's original CD-L1 + CD-L2 (Chamfer only) |
| Architecture | **Unchanged** — ResNet-18 encoder + DGCNN decoder |
| Projection matrix | **Not used** |
| Evaluation | Added F-score @1/2/5mm + HD95 to match PPC v12.3 comparison table |

## Comparison Goal
Same dataset → Same AP+LP input → Same 8192 output points → Same metrics  
**Only difference: DGCNN decoder (3D-ReVert) vs Volumetric Lift + Refinement MLP (PPC v12.3)**

In [1]:
"""
3D-ReVert — Adapted for Lumbar Inter-Comparison
================================================
Encoder : ResNet-18 (2-channel input: AP+LP concatenated)
Decoder : DGCNN (original 3D-ReVert decoder — unchanged)
Loss    : CD-L1 (train) + CD-L2 (monitor) — original 3D-ReVert loss
Input   : AP DRR + LP DRR (no projection matrix)
Output  : 8192 point cloud in world coordinates
"""
import os, sys, json, time, random, warnings, csv, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import vtk
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

# -- MIG SLICE PINNING --------------------------------------------------------
# BX2S-Net is running on Device 0 (MIG-b4e4061f-... / GI 7).
# Pin this notebook to Device 1 (MIG-30b6fd2d-... / GI 8) so both run
# simultaneously without interfering. Must be set BEFORE any CUDA call.
# To use a different slice, swap the UUID below for any idle MIG-* from:
#   nvidia-smi -L
os.environ["CUDA_VISIBLE_DEVICES"] = "MIG-30b6fd2d-1b01-5f01-acd3-27c23d25fb39"

# -- GPU MEMORY CAP (MIG-aware) -----------------------------------------------
# On a MIG partition total_memory reports the slice size (~9.7GB), not the
# full physical GPU (80GB). The old code computed 50.0/total_gb, blew past
# 1.0 and got clamped to 0.95 -- zero safety margin on a small slice.
# Fix: target a concrete GB value, auto-capped to 90% of whatever is visible.
TARGET_GPU_MEM_GB = 8.5
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:512")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Visible device memory (MIG slice or full GPU): {total_gb:.2f} GB")
    frac = min(TARGET_GPU_MEM_GB / total_gb, 0.90)
    if TARGET_GPU_MEM_GB > total_gb:
        print(f"  WARNING: requested {TARGET_GPU_MEM_GB:.1f}GB > visible {total_gb:.2f}GB. "
              f"Capping to {frac*total_gb:.2f}GB (90% of visible device).")
    torch.cuda.set_per_process_memory_fraction(frac, 0)
    print(f"  Memory fraction set: {frac:.3f} (~{frac*total_gb:.2f} GB)")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

Device : cuda
GPU    : NVIDIA A100-SXM4-80GB MIG 1g.10gb


In [2]:
# ── PATHS ──────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path("/apps/app/see_all_ai/dataset/Lumbar_Filtered_1037")
SPLIT_FILE  = DATA_ROOT / "dataset_split.json"
PROJECT_DIR = Path("/apps/app/pandu/3drevert_lumbar_comparison")
CKPT_DIR    = PROJECT_DIR / "checkpoints"
LOG_DIR     = PROJECT_DIR / "logs"
RESULTS_DIR = PROJECT_DIR / "results"
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── CONFIG ─────────────────────────────────────────────────────────────────────
IMG_SIZE            = 512
N_POINTS            = 8192          # match PPC v12.3
IN_CHANNELS         = 2             # AP + LP concatenated (changed from original 1)
PRETRAINED_IMAGENET = True

# DGCNN decoder config (original 3D-ReVert values)
LATENT_DIM          = 1024          # ResNet-18 global feature dim
K_NEIGHBORS         = 20            # DGCNN edge conv neighbors
DGCNN_EMBD_DIMS     = [64, 64, 128, 256]  # EdgeConv output dims
DGCNN_LINEAR_DIMS   = [256, 256]    # MLP after graph features

# Training config
BATCH_SIZE          = 1            # reduced from 2: DGCNN KNN materialises
                                   # (B,N,N) distance matrix; B=2,N=8192 OOMs
                                   # on 9.5GB MIG slice by ep ~150
NUM_WORKERS         = 4
EPOCHS              = 300
LR                  = 1e-4
WEIGHT_DECAY        = 1e-4
LR_DECAY_RATE       = 0.7
LR_DECAY_INTERVAL   = 50
LR_CLIP             = 1e-6
GRAD_CLIP           = 1.0
EARLY_STOP_PATIENCE = 60

MAX_EVAL            = 103
CKPT_PATH           = CKPT_DIR / "latest_checkpoint.pth"
BEST_CKPT           = CKPT_DIR / "best_checkpoint.pth"
TRAINING_LOG        = LOG_DIR  / "training_log.json"

print("=" * 65)
print("3D-ReVert — Adapted for Lumbar Inter-Comparison")
print("=" * 65)
for k, v in dict(
    ENCODER     = f"ResNet-18 (in_channels={IN_CHANNELS}: AP+LP)",
    DECODER     = "DGCNN (original 3D-ReVert, unchanged)",
    N_POINTS    = N_POINTS,
    LOSS        = "CD-L1 (train) + CD-L2 (monitor) — 3D-ReVert original",
    PROJ_MATRIX = "NOT USED",
    BATCH_SIZE  = BATCH_SIZE,
    LR          = f"{LR} (step decay ×{LR_DECAY_RATE} every {LR_DECAY_INTERVAL} ep)",
    EPOCHS      = EPOCHS,
).items():
    print(f"  {k:<18} = {v}")

3D-ReVert — Adapted for Lumbar Inter-Comparison
  ENCODER            = ResNet-18 (in_channels=2: AP+LP)
  DECODER            = DGCNN (original 3D-ReVert, unchanged)
  N_POINTS           = 8192
  LOSS               = CD-L1 (train) + CD-L2 (monitor) — 3D-ReVert original
  PROJ_MATRIX        = NOT USED
  BATCH_SIZE         = 2
  LR                 = 0.0001 (step decay ×0.7 every 50 ep)
  EPOCHS             = 300


## Data Utilities

In [3]:
# ── DATA UTILITIES ─────────────────────────────────────────────────────────────

def load_vtk_points(path):
    r = vtk.vtkPolyDataReader()
    r.SetFileName(str(path))
    r.Update()
    p = r.GetOutput()
    return np.array([p.GetPoint(i) for i in range(p.GetNumberOfPoints())], np.float32)

def save_vtk_points(points, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for pt in points:
        vp.InsertNextPoint(float(pt[0]), float(pt[1]), float(pt[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)):
        vc.InsertNextCell(1)
        vc.InsertCellPoint(i)
    pd = vtk.vtkPolyData()
    pd.SetPoints(vp); pd.SetVerts(vc)
    w = vtk.vtkPolyDataWriter()
    w.SetFileName(str(path)); w.SetInputData(pd)
    w.SetFileTypeToASCII(); w.Write()

def load_drr_image(path, size=IMG_SIZE):
    img = Image.open(path).convert("L")
    if img.size != (size, size):
        img = img.resize((size, size), Image.Resampling.BILINEAR)
    return np.array(img, np.float32) / 255.0

def load_split(p=SPLIT_FILE):
    with open(p) as f:
        d = json.load(f)
    return d["train"], d["val"], d["test"]

def append_log(path, rec):
    log = {"records": []}
    if path.exists():
        with open(path) as f:
            log = json.load(f)
    log["records"].append(rec)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(log, f, indent=2)
    tmp.replace(path)

def pts_to_local(pts, c, s):
    return (pts - c[None]) / (s[None] + 1e-6)

def local_to_world(pts, c, s):
    """pts: (N,3) numpy, c/s: (3,) numpy"""
    return pts * s[None] + c[None]

def compute_scale(gt):
    e = (gt.max(0) - gt.min(0)).astype(np.float32)
    s = e * 0.55 + np.array([20., 20., 30.], np.float32)
    return np.maximum(s, np.array([50., 50., 80.], np.float32))

print("Data utilities defined.")

Data utilities defined.


## Dataset — LumbarDataset (AP + LP, 2-channel)

**Changed from original 3D-ReVert:** Loads AP + LP DRRs and stacks them as a 2-channel image `(2, H, W)`. No projection matrix is used. Everything else follows the original 3D-ReVert data interface (`image`, `points`).

In [4]:
# ── LUMBAR DATASET (changed from original 3D-ReVert dataset) ──────────────────
# Key changes vs original:
#   - Loads 2 views: AP (drr_AP_0.png) + LP (drr_LP_90.png)
#   - Stacks as 2-channel image (2, H, W) — dict key 'image' kept same as original
#   - Ground truth from gt_ppc.vtk in world coordinates
#   - No projection matrix used
#   - Normalizes each channel separately with ImageNet mean/std

# Per-channel normalization mean/std (grayscale duplicated for 2 channels)
NORM_MEAN = [0.485, 0.485]   # channel 0 = AP, channel 1 = LP
NORM_STD  = [0.229, 0.229]


class LumbarDataset(Dataset):
    """
    Adapted dataset for 3D-ReVert inter-comparison.

    Returns the same dict format as original 3D-ReVert:
        'image'  : (2, 512, 512) float tensor  — AP+LP stacked (changed: was 1 or 3 ch)
        'points' : (N_POINTS, 3) float tensor  — world-coord point cloud

    Extra keys for evaluation denormalization (not used by model):
        'center' : (3,) float tensor
        'scale'  : (3,) float tensor
        'patient_id' : str
    """
    def __init__(self, ids, root=DATA_ROOT, aug=False):
        self.ids  = ids
        self.root = Path(root)
        self.aug  = aug
        self.norm = transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]
        d   = self.root / pid

        # Load AP and LP DRR images
        dap = load_drr_image(d / "AP_0"  / "drr_AP_0.png")   # (H, W) float32
        dlp = load_drr_image(d / "LP_90" / "drr_LP_90.png")  # (H, W) float32

        # Basic horizontal flip augmentation
        if self.aug and np.random.rand() < 0.5:
            dap = dap[:, ::-1].copy()
            dlp = dlp[:, ::-1].copy()

        # Stack AP + LP as 2-channel image (changed: original was single-channel)
        img_2ch = np.stack([dap, dlp], axis=0)                   # (2, H, W)
        img_t   = self.norm(torch.from_numpy(img_2ch).float())   # (2, H, W) normalized

        # Load ground truth point cloud
        gt = load_vtk_points(d / "gt_ppc.vtk")   # (M, 3) world coords

        # Store center and scale for denormalization at test time
        c = gt.mean(0).astype(np.float32)
        s = compute_scale(gt)

        # Subsample / oversample to exactly N_POINTS
        n   = len(gt)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))
        gt_sel = gt[sel]   # (N_POINTS, 3) world coords

        return {
            # Keys matching original 3D-ReVert interface
            "image":      img_t,                                        # (2, 512, 512)
            "points":     torch.from_numpy(gt_sel).float(),             # (N_POINTS, 3) world
            # Extra keys for evaluation only
            "center":     torch.from_numpy(c).float(),                  # (3,)
            "scale":      torch.from_numpy(s).float(),                  # (3,)
            "patient_id": pid,
        }


def collate_fn(b):
    out = {}
    for k in b[0]:
        vals = [x[k] for x in b]
        out[k] = torch.stack(vals, 0) if isinstance(vals[0], torch.Tensor) else vals
    return out


train_ids, val_ids, test_ids = load_split()
print(f"Split: train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}")

# Verify one sample loads correctly
_ds = LumbarDataset(train_ids[:1])
_s  = _ds[0]
print(f"Sample check — image: {_s['image'].shape}  points: {_s['points'].shape}")
print(f"  center: {_s['center'].numpy().round(1)}  scale: {_s['scale'].numpy().round(1)}")
del _ds, _s

Split: train=829  val=103  test=105
Sample check — image: torch.Size([2, 512, 512])  points: torch.Size([8192, 3])
  center: [  18.  -166.4 -906. ]  scale: [ 73.1  74.2 118.5]


## Model — ResNet-18 Encoder + DGCNN Decoder

**Original 3D-ReVert architecture, with one change:** `in_channels` of the first conv layer changed from 1 to 2 to accept the stacked AP+LP image.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# ENCODER — ResNet-18 (in_channels changed 1→2, everything else original)
# ══════════════════════════════════════════════════════════════════════════════

class ResNet18Encoder(nn.Module):
    """
    Original 3D-ReVert encoder.
    Only change: conv1 in_channels = 2 (AP+LP) instead of 1 (single DRR).
    Outputs a global feature vector of dim LATENT_DIM.
    """
    def __init__(self, in_ch=IN_CHANNELS, latent_dim=LATENT_DIM,
                 pretrained=PRETRAINED_IMAGENET):
        super().__init__()
        base = models.resnet18(
            weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        )

        # ── Changed: first conv accepts 2 channels instead of 1 ──
        new_conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)
        if pretrained and in_ch <= 3:
            # Average the pretrained RGB weights over the in_ch channels
            with torch.no_grad():
                new_conv1.weight[:] = base.conv1.weight.mean(1, keepdim=True).expand(
                    -1, in_ch, -1, -1
                ) / in_ch
        base.conv1 = new_conv1
        # ── End change ──

        # Remove the classification head; keep everything up to avgpool
        self.features = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4,
            base.avgpool,
        )
        # Project from ResNet-18's 512-dim pool to LATENT_DIM
        self.fc = nn.Linear(512, latent_dim)

    def forward(self, x):
        """
        x : (B, 2, H, W)  — stacked AP + LP
        returns : (B, LATENT_DIM)
        """
        feat = self.features(x).flatten(1)   # (B, 512)
        return self.fc(feat)                  # (B, LATENT_DIM)


print("ResNet-18 encoder defined (in_channels=2).")

ResNet-18 encoder defined (in_channels=2).


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# DECODER — DGCNN (original 3D-ReVert, completely unchanged)
# ══════════════════════════════════════════════════════════════════════════════

def knn(x, k):
    """
    Find k-nearest neighbours for each point.
    x   : (B, C, N)
    returns idx : (B, N, k)
    """
    inner   = -2 * torch.bmm(x.transpose(2, 1), x)       # (B, N, N)
    xx      = (x ** 2).sum(1, keepdim=True)               # (B, 1, N)
    dist    = xx + inner + xx.transpose(2, 1)             # (B, N, N)
    idx     = dist.topk(k=k, dim=-1, largest=False)[1]    # (B, N, k)
    return idx


def get_graph_feature(x, k, idx=None):
    """
    Construct edge features for DGCNN.
    x   : (B, C, N)
    returns : (B, 2C, N, k)
    """
    B, C, N = x.shape
    if idx is None:
        idx = knn(x, k=k)   # (B, N, k)

    # Gather neighbours
    idx_base = torch.arange(0, B, device=x.device).view(-1, 1, 1) * N
    idx_flat  = (idx + idx_base).view(-1)                          # (B*N*k,)
    x_t = x.transpose(2, 1).contiguous().view(B * N, C)           # (B*N, C)
    neighbors = x_t[idx_flat].view(B, N, k, C).permute(0, 3, 1, 2)  # (B,C,N,k)

    # Relative feature = neighbour − centre, concatenated with centre
    x_expanded = x.unsqueeze(3).expand_as(neighbors)              # (B,C,N,k)
    edge_feat   = torch.cat([neighbors - x_expanded, x_expanded], dim=1)  # (B,2C,N,k)
    return edge_feat


class EdgeConv(nn.Module):
    """Single EdgeConv block from DGCNN."""
    def __init__(self, in_ch, out_ch, k):
        super().__init__()
        self.k   = k
        self.net = nn.Sequential(
            nn.Conv2d(in_ch * 2, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        """
        x : (B, in_ch, N)
        returns : (B, out_ch, N)
        """
        edge  = get_graph_feature(x, k=self.k)   # (B, 2*in_ch, N, k)
        out   = self.net(edge)                    # (B, out_ch, N, k)
        return out.max(dim=-1)[0]                 # (B, out_ch, N) — max-pool over k


class DGCNNDecoder(nn.Module):
    """
    Original 3D-ReVert DGCNN decoder — completely unchanged.

    Takes a global latent vector z (B, LATENT_DIM) and a coarse seed
    point set, then refines via stacked EdgeConv layers to produce the
    final point cloud (B, N_POINTS, 3).

    Architecture:
      1. Expand z to (B, LATENT_DIM, N_POINTS) — tiled across points
      2. Seed coords (B, 3, N_POINTS) concatenated
      3. Four EdgeConv blocks with progressive feature dims
      4. Global max-pool aggregated back and concatenated per-point
      5. Linear MLP → 3D offsets → final point cloud
    """
    def __init__(self, latent_dim=LATENT_DIM, n_pts=N_POINTS,
                 k=K_NEIGHBORS, embd_dims=DGCNN_EMBD_DIMS,
                 linear_dims=DGCNN_LINEAR_DIMS):
        super().__init__()
        self.n_pts     = n_pts
        self.latent_dim = latent_dim

        # Seed generator: latent → coarse point set
        self.seed_mlp = nn.Sequential(
            nn.Linear(latent_dim, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, n_pts * 3),
        )

        # Input to EdgeConv: seed coords (3) + latent tiled (latent_dim)
        in_ch = 3 + latent_dim

        self.ec1 = EdgeConv(in_ch,        embd_dims[0], k)
        self.ec2 = EdgeConv(embd_dims[0], embd_dims[1], k)
        self.ec3 = EdgeConv(embd_dims[1], embd_dims[2], k)
        self.ec4 = EdgeConv(embd_dims[2], embd_dims[3], k)

        # Aggregate feature dim: sum of all EdgeConv outputs
        agg_dim = sum(embd_dims)   # 64+64+128+256 = 512

        # Per-point MLP to predict 3D offset from seed
        self.refine_mlp = nn.Sequential(
            nn.Conv1d(agg_dim, linear_dims[0], 1, bias=False),
            nn.BatchNorm1d(linear_dims[0]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(linear_dims[0], linear_dims[1], 1, bias=False),
            nn.BatchNorm1d(linear_dims[1]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(linear_dims[1], 3, 1),
        )

    def forward(self, z):
        """
        z       : (B, LATENT_DIM)   global image feature from encoder
        returns : (B, N_POINTS, 3)  predicted point cloud
        """
        B = z.shape[0]

        # 1. Generate coarse seed points from latent vector
        seed = self.seed_mlp(z).view(B, self.n_pts, 3)   # (B, N, 3)
        seed = torch.tanh(seed)                           # bound to [-1, 1]

        # 2. Tile global feature across all seed points
        seed_t  = seed.transpose(1, 2).contiguous()       # (B, 3, N)
        z_tiled = z.unsqueeze(2).expand(-1, -1, self.n_pts)  # (B, LATENT_DIM, N)

        # 3. Concatenate: (B, 3+LATENT_DIM, N)
        x = torch.cat([seed_t, z_tiled], dim=1)

        # 4. Stacked EdgeConv
        x1 = self.ec1(x)                   # (B, 64, N)
        x2 = self.ec2(x1)                  # (B, 64, N)
        x3 = self.ec3(x2)                  # (B, 128, N)
        x4 = self.ec4(x3)                  # (B, 256, N)

        # 5. Concatenate all levels: (B, 512, N)
        x_cat = torch.cat([x1, x2, x3, x4], dim=1)

        # 6. Per-point refinement MLP → 3D offset
        delta = self.refine_mlp(x_cat).transpose(1, 2)   # (B, N, 3)

        # 7. Add offset to seed
        pred = seed + 0.1 * torch.tanh(delta)            # (B, N, 3)
        return pred


print("DGCNN decoder defined (original 3D-ReVert, unchanged).")

DGCNN decoder defined (original 3D-ReVert, unchanged).


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# FULL MODEL — 3D-ReVert adapted
# ══════════════════════════════════════════════════════════════════════════════

class ReVertLumbar(nn.Module):
    """
    3D-ReVert adapted for Lumbar inter-comparison.

    Forward interface:
        image  : (B, 2, H, W)  — AP+LP stacked DRR

    Output dict (matches PPC v12.3 output format for evaluation):
        pred_world : (B, N, 3) — predicted point cloud in world coordinates

    NOTE: The model works in NORMALISED local space [-1,1] internally.
    Denormalization (local → world) is done post-hoc using center/scale
    from the dataset — see the evaluation cell.
    """
    def __init__(self):
        super().__init__()
        self.encoder = ResNet18Encoder()
        self.decoder = DGCNNDecoder()

    def forward(self, image):
        """
        image : (B, 2, H, W)
        returns dict with 'pred_local' (B, N, 3) in normalised space
        """
        z    = self.encoder(image)       # (B, LATENT_DIM)
        pred = self.decoder(z)           # (B, N_POINTS, 3) in [-1,1] approx
        return {"pred_local": pred}


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


_m = ReVertLumbar()
enc_p = count_params(_m.encoder)
dec_p = count_params(_m.decoder)
print(f"ReVertLumbar total params : {(enc_p+dec_p)/1e6:.1f} M")
print(f"  Encoder (ResNet-18)     : {enc_p/1e6:.1f} M")
print(f"  Decoder (DGCNN)         : {dec_p/1e6:.1f} M")
del _m

ReVertLumbar total params : 38.4 M
  Encoder (ResNet-18)     : 11.7 M
  Decoder (DGCNN)         : 26.7 M


## Loss Functions — Original 3D-ReVert (CD-L1 + CD-L2)

3D-ReVert trains with **CD-L1 only**. CD-L2 is computed for monitoring but not backpropagated. Both are computed in **normalised local space** (ground truth is also normalised before computing loss).

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# LOSS — Original 3D-ReVert: CD-L1 (train) + CD-L2 (monitor)
# ══════════════════════════════════════════════════════════════════════════════
# GT is normalised to local space before computing loss, same as original.

def pairwise_dist(pred, gt):
    """
    Compute pairwise squared L2 distances.
    pred, gt : (B, N, 3)
    returns  : (B, N_pred, N_gt)
    """
    p2 = (pred ** 2).sum(-1, keepdim=True)              # (B, N, 1)
    g2 = (gt   ** 2).sum(-1).unsqueeze(1)               # (B, 1, M)
    return (p2 + g2 - 2 * torch.bmm(pred, gt.transpose(1, 2))).clamp_min(0)  # (B, N, M)


CD_EPS = 1e-8

def calc_cd(pred, gt):
    d2    = pairwise_dist(pred, gt)        # (B, N, M) squared, >= 0
    d_p2g = d2.min(2)[0]                   # (B, N) squared
    d_g2p = d2.min(1)[0]                   # (B, M) squared

    # CD-L1: eps INSIDE sqrt -> finite gradient even when a pred point
    # coincides with a GT point (this was the NaN source)
    cd_l1 = ((d_p2g + CD_EPS).sqrt().mean(1) +
             (d_g2p + CD_EPS).sqrt().mean(1)) * 0.5      # (B,)

    cd_l2 = (d_p2g.mean(1) + d_g2p.mean(1)) * 0.5         # (B,) squared, no sqrt
    return cd_l1, cd_l2


def normalise_gt_batch(gt_world, center, scale):
    """
    Convert world-space GT to local normalised space for loss computation.
    gt_world : (B, N, 3)
    center   : (B, 3)
    scale    : (B, 3)
    """
    c = center.unsqueeze(1)   # (B, 1, 3)
    s = scale.unsqueeze(1)    # (B, 1, 3)
    return (gt_world - c) / (s + 1e-6)


print("Loss functions defined (CD-L1 train + CD-L2 monitor — original 3D-ReVert).")

Loss functions defined (CD-L1 train + CD-L2 monitor — original 3D-ReVert).


## Training Loop

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════

def to_dev(b):
    return {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in b.items()}


# ── Build model ───────────────────────────────────────────────────────────────
model = ReVertLumbar().to(device)
print(f"Model params: {count_params(model)/1e6:.1f} M")

# Separate encoder / decoder params for differential LR
enc_params   = list(model.encoder.parameters())
enc_ids      = {id(p) for p in enc_params}
other_params = [p for p in model.parameters() if id(p) not in enc_ids]

optimizer = torch.optim.Adam([
    {"params": enc_params,   "lr": LR * 0.1},   # encoder lower LR (pretrained)
    {"params": other_params, "lr": LR},
], weight_decay=WEIGHT_DECAY)

# ── Data loaders ──────────────────────────────────────────────────────────────
train_ds = LumbarDataset(train_ids, aug=True)
val_ds   = LumbarDataset(val_ids,   aug=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)

print(f"Train: {len(train_ds)} samples → {len(train_loader)} batches/ep")
print(f"Val  : {len(val_ds)} samples → {len(val_loader)} batches/ep")


def save_ckpt(path, ep, best_cd, hist):
    tmp = path.with_suffix(".tmp")
    torch.save({
        "model":        model.state_dict(),
        "optimizer":    optimizer.state_dict(),
        "epoch":        ep,
        "best_cd_l1":   best_cd,
        "history":      hist,
        "config":       {"version": "3DReVert_lumbar_comparison",
                         "n_points": N_POINTS, "in_channels": IN_CHANNELS},
    }, tmp)
    tmp.replace(path)
    print(f"  Saved → {path.name}  (ep {ep+1})")


def load_ckpt(path):
    st = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(st["model"])
    optimizer.load_state_dict(st["optimizer"])
    ep   = st["epoch"] + 1
    bc   = st.get("best_cd_l1", float("inf"))
    hist = st.get("history", [])
    print(f"  Resumed from ep {ep} | best CD-L1 = {bc:.5f}")
    return ep, bc, hist


# ── LR step-decay (original 3D-ReVert schedule) ───────────────────────────────
def adjust_lr(optimizer, epoch):
    if LR_DECAY_INTERVAL and epoch > 0 and epoch % LR_DECAY_INTERVAL == 0:
        for pg in optimizer.param_groups:
            pg["lr"] = max(pg["lr"] * LR_DECAY_RATE, LR_CLIP)
        cur_lrs = [pg["lr"] for pg in optimizer.param_groups]
        print(f"  LR adjusted → {cur_lrs}")


start_epoch, best_cd_l1, history, no_improve = 0, float("inf"), [], 0

if CKPT_PATH.exists():
    print(f"Found checkpoint: {CKPT_PATH.name}")
    start_epoch, best_cd_l1, history = load_ckpt(CKPT_PATH)
else:
    print("No checkpoint — starting fresh.")

print(f"{'='*65}")
print(f"3D-ReVert training from ep {start_epoch+1}/{EPOCHS}")
print(f"Loss: CD-L1 (train) + CD-L2 (monitor)")
print(f"{'='*65}")

Model params: 38.4 M
Train: 829 samples → 415 batches/ep
Val  : 103 samples → 52 batches/ep
Found checkpoint: latest_checkpoint.pth
  Resumed from ep 149 | best CD-L1 = 0.02820
3D-ReVert training from ep 150/300
Loss: CD-L1 (train) + CD-L2 (monitor)


In [10]:
# ── Main training loop ────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS):
    adjust_lr(optimizer, epoch)
    # Clear allocator fragmentation that builds up over many epochs.
    # On a 9.5GB MIG slice the DGCNN KNN op needs large contiguous blocks;
    # fragmentation from prior epochs causes OOM even when total free > needed.
    torch.cuda.empty_cache()
    # ── Train ──
    model.train()
    t0 = time.time()
    total_l1, total_l2, nb, nan_count = 0.0, 0.0, 0, 0
    pbar = tqdm(enumerate(train_loader, 1), total=len(train_loader),
                desc=f'Ep {epoch+1:3d}/{EPOCHS}', leave=True, ncols=120)
    for bi, batch in pbar:
        batch = to_dev(batch)
        # Forward
        out = model(batch["image"])
        pred_local = out["pred_local"]   # (B, N, 3) in local space
        # Normalise GT to local space for loss
        gt_local = normalise_gt_batch(
            batch["points"], batch["center"], batch["scale"]
        ).clamp(-1, 1)

        # Guard 1: skip batch if model output / GT is already non-finite
        if not (torch.isfinite(pred_local).all() and torch.isfinite(gt_local).all()):
            nan_count += 1
            if nan_count > 50:
                print(f"  ⚠ Too many NaN ({nan_count}), stopping epoch"); break
            continue

        # CD-L1 (training loss) + CD-L2 (monitor only)
        cd_l1, cd_l2 = calc_cd(pred_local, gt_local)
        loss = cd_l1.mean()
        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            if nan_count > 50:
                print(f"  ⚠ Too many NaN ({nan_count}), stopping epoch"); break
            continue

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        # Guard 2: never step on non-finite grads — prevents one bad batch
        # from turning all weights into NaN (the root cause of the prior run)
        if not torch.isfinite(gnorm):
            optimizer.zero_grad(set_to_none=True)
            nan_count += 1
            continue

        optimizer.step()
        l1_val = cd_l1.mean().item()
        l2_val = cd_l2.mean().item()
        total_l1 += l1_val; total_l2 += l2_val; nb += 1
        if bi % 50 == 0 or bi == len(train_loader):
            pbar.set_postfix_str(
                f"cd_l1={total_l1/nb:.5f}  cd_l2={total_l2/nb:.5f}  nan={nan_count}"
            )
    elapsed = time.time() - t0
    avg_l1 = total_l1 / max(1, nb)
    avg_l2 = total_l2 / max(1, nb)
    # ── Validate ──
    model.eval()
    val_l1_list, val_l2_list = [], []
    nd = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='  Val', leave=False, ncols=100):
            if nd >= MAX_EVAL: break
            batch = to_dev(batch)
            out = model(batch["image"])
            pred_local = out["pred_local"]
            gt_local   = normalise_gt_batch(
                batch["points"], batch["center"], batch["scale"]
            ).clamp(-1, 1)
            cd_l1, cd_l2 = calc_cd(pred_local, gt_local)
            for b in range(pred_local.shape[0]):
                if nd >= MAX_EVAL: break
                # skip any non-finite per-sample value so one bad sample
                # can't turn the whole val mean into NaN
                v1, v2 = cd_l1[b].item(), cd_l2[b].item()
                if not (np.isfinite(v1) and np.isfinite(v2)):
                    nd += 1; continue
                val_l1_list.append(v1)
                val_l2_list.append(v2)
                nd += 1
    val_l1 = float(np.mean(val_l1_list)) if val_l1_list else float("nan")
    val_l2 = float(np.mean(val_l2_list)) if val_l2_list else float("nan")
    print(f"\n[Ep {epoch+1}] {elapsed:.0f}s  "
          f"Train CD-L1={avg_l1:.5f} CD-L2={avg_l2:.5f}  "
          f"Val CD-L1={val_l1:.5f} CD-L2={val_l2:.5f}")
    rec = {
        "epoch":      epoch + 1,
        "train_l1":   avg_l1,
        "train_l2":   avg_l2,
        "val_l1":     val_l1,
        "val_l2":     val_l2,
        "nan_skips":  nan_count,
    }
    history.append(rec)
    append_log(TRAINING_LOG, rec)
    save_ckpt(CKPT_PATH, epoch, best_cd_l1, history)
    if val_l1 < best_cd_l1:
        best_cd_l1 = val_l1
        no_improve = 0
        save_ckpt(BEST_CKPT, epoch, best_cd_l1, history)
        print(f"  ✓ New best Val CD-L1: {best_cd_l1:.5f}")
    else:
        no_improve += 1
        if no_improve >= EARLY_STOP_PATIENCE:
            print(f"  Early stop: {no_improve} epochs without improvement"); break
print(f"Done. Best Val CD-L1: {best_cd_l1:.5f}")

Ep 150/300:   0%|                                                                               | 0/415 [00:00<?, ?it/s]


RuntimeError: NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/opt/pytorch/pytorch/c10/cuda/CUDACachingAllocator.cpp":1150, please report a bug to PyTorch. 

In [13]:

torch.cuda.device_count() 

1

## Test Evaluation — With World-Space Denormalisation

The model predicts in **normalised local space [-1, 1]**. Before computing Chamfer/F-score/HD95 for comparison with PPC v12.3, we denormalise predictions back to **world coordinates (mm)** using the per-sample `center` and `scale` stored by the dataset.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TEST EVALUATION — World-space metrics for PPC v12.3 comparison
# ══════════════════════════════════════════════════════════════════════════════
import gc, torch

# -- Free ALL GPU memory left over from training ------------------------------
try: del optimizer
except NameError: pass
try: del model                        # critical — free training model from GPU
except NameError: pass

torch.cuda.synchronize()
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()             # second pass after gc frees dangling refs

if torch.cuda.is_available():
    free  = torch.cuda.mem_get_info(0)[0] / 1e9
    total = torch.cuda.mem_get_info(0)[1] / 1e9
    print(f"GPU memory free before test load: {free:.2f} / {total:.2f} GB")

# -- Rebuild model fresh, load checkpoint safely via CPU ----------------------
def chamfer_np(pred, gt):
    """
    Numpy Chamfer + F-score + HD95 — identical to PPC v12.3's chamfer_np.
    pred, gt : (N, 3) numpy arrays in world coordinates (mm)
    """
    pred = np.asarray(pred, np.float32)
    gt   = np.asarray(gt,   np.float32)
    d    = np.linalg.norm(pred[:, None] - gt[None], axis=-1)
    dp   = d.min(1)
    dg   = d.min(0)

    def fscore(th):
        p = (dp <= th).mean()
        r = (dg <= th).mean()
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    return {
        "chamfer_mm":   float(0.5 * (dp.mean() + dg.mean())),
        "fscore_1mm":   float(fscore(1)),
        "fscore_2mm":   float(fscore(2)),
        "fscore_5mm":   float(fscore(5)),
        "hausdorff_95": float(max(np.percentile(dp, 95), np.percentile(dg, 95))),
    }

print("Test evaluation (best checkpoint)...")
if BEST_CKPT.exists():
    st = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)  # CPU first
    model = ReVertLumbar().cpu()
    model.load_state_dict(st["model"])
    print(f"  Loaded best checkpoint from ep {st['epoch']+1} "
          f"(val CD-L1 = {st['best_cd_l1']:.5f})")
    del st                            # free checkpoint dict before moving to GPU
    gc.collect()
    model = model.to(device)
else:
    print("  WARNING: No best checkpoint found — using current model state.")
    model = ReVertLumbar().to(device)

if torch.cuda.is_available():
    free = torch.cuda.mem_get_info(0)[0] / 1e9
    print(f"GPU memory free after model load:  {free:.2f} / {total:.2f} GB")

model.eval()

# -- Test dataloader ----------------------------------------------------------
test_ds     = LumbarDataset(test_ids, aug=False)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         collate_fn=collate_fn)

all_metrics = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='  Test', ncols=100):
        batch = to_dev(batch)
        B = batch["image"].shape[0]

        out        = model(batch["image"])
        pred_local = out["pred_local"].cpu().numpy()   # (B, N, 3)

        center = batch["center"].cpu().numpy()         # (B, 3)
        scale  = batch["scale"].cpu().numpy()          # (B, 3)
        gt_w   = batch["points"].cpu().numpy()         # (B, N, 3) world
        pids   = batch["patient_id"]

        for b in range(B):
            pred_world = local_to_world(pred_local[b], center[b], scale[b])
            m = chamfer_np(pred_world, gt_w[b])
            m["patient_id"] = pids[b]
            all_metrics.append(m)
            save_vtk_points(pred_world, RESULTS_DIR / f"{pids[b]}_3drevert_pred.vtk")

# -- Print results ------------------------------------------------------------
print(f"\n{'='*65}")
print(f"3D-ReVert TEST RESULTS — {len(all_metrics)} patients — World Space (mm)")
print(f"{'='*65}")
for key, lbl in [("chamfer_mm",   "Chamfer (mm) "),
                 ("fscore_1mm",   "F-score @1mm "),
                 ("fscore_2mm",   "F-score @2mm "),
                 ("fscore_5mm",   "F-score @5mm "),
                 ("hausdorff_95", "HD95 (mm)    ")]:
    vals = [m[key] for m in all_metrics]
    print(f"  {lbl}  mean={np.mean(vals):.3f}  std={np.std(vals):.3f}  "
          f"median={np.median(vals):.3f}  p95={np.percentile(vals,95):.3f}")

# -- Save CSV -----------------------------------------------------------------
csv_path = RESULTS_DIR / "test_results_3drevert_lumbar.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=[
        "patient_id", "chamfer_mm",
        "fscore_1mm", "fscore_2mm", "fscore_5mm", "hausdorff_95"
    ])
    w.writeheader()
    w.writerows(all_metrics)
print(f"\nCSV saved → {csv_path}")


## Inter-Comparison Table

Run this cell after you also have PPC v12.3 results saved. It prints a side-by-side table.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INTER-COMPARISON TABLE — 3D-ReVert vs PPC v12.3
# ══════════════════════════════════════════════════════════════════════════════
# Reads:
#   - 3D-ReVert results from this run's CSV
#   - PPC v12.3 results from its CSV (adjust path if needed)

import csv as _csv

PPC_CSV = Path("/apps/app/pandu/ppc_network_v12_3/results/test_results_v12_3_tta.csv")

def load_csv_metrics(path):
    rows = []
    with open(path) as f:
        for row in _csv.DictReader(f):
            rows.append({k: float(v) if k != "patient_id" else v
                         for k, v in row.items()})
    return rows


revert_metrics = load_csv_metrics(csv_path)

print(f"{'='*75}")
print(f"{'Metric':<22} {'3D-ReVert':>18} {'PPC v12.3':>18} {'Δ (PPC−ReVert)':>16}")
print(f"{'':22} {'(DGCNN decoder)':>18} {'(Vol+MLP decoder)':>18}")
print(f"{'-'*75}")

metric_labels = [
    ("chamfer_mm",   "Chamfer (mm)  ↓"),
    ("fscore_1mm",   "F-score @1mm  ↑"),
    ("fscore_2mm",   "F-score @2mm  ↑"),
    ("fscore_5mm",   "F-score @5mm  ↑"),
    ("hausdorff_95", "HD95 (mm)     ↓"),
]

if PPC_CSV.exists():
    ppc_metrics = load_csv_metrics(PPC_CSV)
    for key, lbl in metric_labels:
        rv = np.mean([m[key] for m in revert_metrics])
        pv = np.mean([m[key] for m in ppc_metrics])
        delta = pv - rv
        print(f"  {lbl:<20} {rv:>18.3f} {pv:>18.3f} {delta:>+16.3f}")
else:
    print("  PPC v12.3 CSV not found — showing 3D-ReVert results only.")
    for key, lbl in metric_labels:
        rv = np.mean([m[key] for m in revert_metrics])
        print(f"  {lbl:<20} {rv:>18.3f}")

print(f"{'='*75}")
print(f"  Patients evaluated: 3D-ReVert = {len(revert_metrics)}", end="")
if PPC_CSV.exists():
    print(f"  |  PPC v12.3 = {len(ppc_metrics)}")
else:
    print()

print("\nNote: Both models trained/tested on same Lumbar_Filtered_1037 dataset.")
print("      Both use AP+LP DRR input.  N_POINTS = 8192.")
print("      Metrics computed in world coordinates (mm).")
print("      Only architectural difference: decoder (DGCNN vs Vol-Lift+MLP).")